In [ ]:
! pip install -q kaggle

In [ ]:
!pip install transformers datasets

In [ ]:
from google.colab import files




In [ ]:
files.upload()

In [ ]:
!pip install torch

In [ ]:
import pandas as pd

In [ ]:
train  = pd.read_csv('train.csv')
test   = pd.read_csv('test.csv')

In [ ]:
train.head()

In [ ]:
test

In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification, AdamW
from torch.utils.data import DataLoader, TensorDataset , RandomSampler, SequentialSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [ ]:
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')

In [ ]:
# Initialize the lemmatizer
lemmatizer = WordNetLemmatizer()

In [ ]:
# DATA CLEANING

In [ ]:
def get_wordnet_pos(word):
    """Map POS tag to first character lemmatize() accepts"""
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": nltk.corpus.wordnet.ADJ,
                "N": nltk.corpus.wordnet.NOUN,
                "V": nltk.corpus.wordnet.VERB,
                "R": nltk.corpus.wordnet.ADV}
    return tag_dict.get(tag, nltk.corpus.wordnet.NOUN)

In [ ]:
def clean_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # Remove user @ references and '#' from tweet
    text = re.sub(r'\@\w+|\#','', text)

    # Remove punctuations and numbers
    text = re.sub('[^a-zA-Z]', ' ', text)

    # Remove single characters
    text = re.sub(r'\s+[a-zA-Z]\s+', ' ', text)

    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text, flags=re.I)

    # Tokenize the text
    tokens = word_tokenize(text)

    # Remove Stopwords
    tokens = [word for word in tokens if word not in stopwords.words('english')]

    # Lemmatize
    tokens = [lemmatizer.lemmatize(word, get_wordnet_pos(word)) for word in tokens]

    return ' '.join(tokens)


In [ ]:
# Apply cleaning to train dataframe
train['cleaned_text'] = train['text'].apply(clean_text)

# Apply cleaning to test dataframe
test['cleaned_text'] = test['text'].apply(clean_text)

In [ ]:
train.head()

In [ ]:
train.isnull().sum()

Using BERT


In [ ]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import torch.optim as optim
import numpy as np

# Set device to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# Initialize the tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Tokenize the text column
def tokenize_texts(texts):
    return tokenizer(list(texts), padding=True, truncation=True, return_tensors='pt', max_length=128)

# Combine 'cleaned_text', 'keywords', and 'location' for input
X = train[['cleaned_text', 'keyword', 'location']].fillna('')  # Filling missing values
encoded_inputs = tokenize_texts(X['cleaned_text'] + ' ' + X['keyword'] + ' ' + X['location'])

# Convert the labels to tensor format
y = torch.tensor(train['target'].values)


In [ ]:
# Split the dataset into training and validation sets
train_inputs, val_inputs, train_labels, val_labels = train_test_split(
    encoded_inputs['input_ids'], y, test_size=0.2, random_state=42)

# Create DataLoaders for batching
train_dataset = TensorDataset(train_inputs, train_labels)
val_dataset = TensorDataset(val_inputs, val_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)


In [ ]:
# Load DistilBERT for sequence classification
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
model.to(device)  # Move the model to GPU (if available)


In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=5e-5)
loss_fn = torch.nn.CrossEntropyLoss()

In [ ]:
# Training loop
epochs = 3  # You can increase this if needed

for epoch in range(epochs):
    model.train()  # Set model to training mode
    total_loss = 0

    for batch in train_loader:
        # Unpack the batch and move inputs/labels to the GPU (or CPU)
        inputs, labels = batch
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()  # Reset gradients

        # Forward pass
        outputs = model(inputs)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()  # Backward pass
        optimizer.step()  # Update parameters

        total_loss += loss.item()

    print(f"Epoch {epoch + 1}, Loss: {total_loss / len(train_loader)}")


In [ ]:
model.eval()  # Set model to evaluation mode

total_eval_loss = 0
correct = 0
total = 0

with torch.no_grad():
    for batch in val_loader:
        inputs, labels = batch
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        loss = loss_fn(outputs.logits, labels)
        total_eval_loss += loss.item()

        # Calculate accuracy
        predictions = torch.argmax(outputs.logits, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

print(f"Validation Loss: {total_eval_loss / len(val_loader)}")
print(f"Validation Accuracy: {correct / total}")


In [ ]:
# Assume you have a DataFrame 'test' with the same structure as 'train'
# Fill missing values in the test dataset
test.fillna('', inplace=True)  # Replace NaNs with empty strings

# Tokenize the test data
encoded_test_inputs = tokenizer(
    list(test['cleaned_text'] + ' ' + test['keyword'] + ' ' + test['location']),
    padding=True,
    truncation=True,
    return_tensors='pt',
    max_length=128
)

# Convert to tensors and move to device
test_inputs = encoded_test_inputs['input_ids'].to(device)


In [ ]:
# Create a DataLoader for the test dataset
test_dataset = TensorDataset(test_inputs)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


In [ ]:
model.eval()  # Set model to evaluation mode

predictions = []

with torch.no_grad():
    for batch in test_loader:
        inputs = batch[0].to(device)  # Move inputs to device
        outputs = model(inputs)

        # Get predicted class
        preds = torch.argmax(outputs.logits, dim=1)
        predictions.extend(preds.cpu().numpy())  # Move back to CPU and convert to numpy array

# Convert predictions to a DataFrame or Series
predictions_df = pd.Series(predictions, name='predictions')


In [ ]:
test['predictions'] = predictions_df
test[['id', 'predictions']].to_csv('disaster_tweet_predictions.csv', index=False)


In [ ]:
test.head()

In [ ]:
from google.colab import files
files.download('disaster_tweet_predictions.csv')

In [ ]:
# Assuming you already have the predictions in the DataFrame
test['target'] = predictions_df  # Rename 'predictions' to 'target'

# Save the modified DataFrame to a CSV file with the new column name
test[['id', 'target']].to_csv('submissions_BERT.csv', index=False)


In [ ]:
files.download('submissions_BERT.csv')